### Instruction Dataset selber bauen

Da wir keinen Zugriff auf den RecipeNLG Alpaca Instruction Dataset haben, wollen wir aus dem RecipeNLG Datensatz einen eigenen Instruction Datensatz generieren.



#### Datensatz aus der Cloud importieren

In [ ]:
import pandas as pd

# Reduzierter Datensatz mit 100k Zeilen zum Testen
# cloud_url = "https://syncandshare.lrz.de/dl/fiHE8nDPcb4nww3VCn4QmN/reduced_dataset_100k.csv"
# Die folgende Zeile einkommentieren, um den vollständigen Datensatz zu verwenden
cloud_url = "https://syncandshare.lrz.de/dl/finR5gtyQx3FNL5P2hz6H/full_dataset.csv"

df = None
try:
    print("Lade Datensatz aus der Cloud...")
    df = pd.read_csv(cloud_url)
    print("Datensatz erfolgreich geladen!\n")
    print("Info:")
    print(df.info())
    print("\nAnfang (Head) des Datensatzes:")
    print(df.head())
except Exception as e:
    print(f"Ein Fehler ist beim Laden des Datensatzes aufgetreten: {e}")

Lade Datensatz aus der Cloud...


#### Konfiguration & Templates

Wir erzeugen drei Instruktions-Typen, deren **Output** exakt dem Pre-Training-Format
(Recipe: / Ingredients: / Directions:) entspricht, damit das Instruction-Tuning auf
dem im Pre-Training gelernten Format aufsetzt:

1. title_to_recipe: nur der Titel ist gegeben.
2. ingredients_to_recipe: nur (Teil-)Zutaten gegeben ("Kühlschrank-Fall").
3. title_ingredients_to_recipe: Titel + (Teil-)Zutaten gegeben.

Kühlschrank-Idee / Ingredient-Drop: Im Prompt stehen die Kern-Zutaten aus der NER-Spalte
(z. B. brown sugar, milk, vanilla). Davon wird pro Beispiel ein zufälliger Teil weggelassen
(p ~ Uniform(0, 0.5)), der Output enthält aber immer die vollständige Liste inkl. Mengen.
So lernt das Modell zwei Dinge gleichzeitig: Kern-Zutaten zu Zutaten mit Mengenangaben ausbauen
und fehlende Zutaten passend ergänzen.

In [ ]:
import os
import json
import random

import tiktoken

# ----------------------------- Konfiguration --------------------------------
SEED        = 123
N_TRAIN     = 15_000
N_VAL       = 1_000
N_TEST      = 1_000
MAX_TOKENS  = 512          # 512 = eigenes Pre-Training-Modell, 1024 = OpenAI gpt2-medium
DROP_P_MAX  = 0.5          # p ~ Uniform(0, DROP_P_MAX) pro Beispiel
OUT_DIR     = "datasets/instruction"
TASKS       = ["title_to_recipe", "ingredients_to_recipe", "title_ingredients_to_recipe"]

tokenizer = tiktoken.get_encoding("gpt2")

# --------------------------------- Templates --------------------------------
# Platzhalter:  {title}, {ingredients} (Komma-getrennt), {ingredients_block} (eine pro Zeile)
# ~15 pro Task, gemischte Register (casual / imperativ / Frage) + jeweils ~3 strukturierte.

TITLE_TO_RECIPE_TEMPLATES = [
    "I want to cook {title} tonight — can you give me the full recipe?",
    "Craving {title}. How do I make it?",
    "I'd love to try making {title}. Walk me through it.",
    "Write a full recipe for {title}.",
    "Give me the ingredients and steps for {title}.",
    "Put together a complete recipe for {title}.",
    "Share a recipe for {title} with ingredients and directions.",
    "How do I make {title}?",
    "How is {title} prepared?",
    "Can you show me how to cook {title}?",
    "What's a good recipe for {title}?",
    "Whats the easiest way to make {title}?",
    "Dish: {title}\nWrite the recipe with ingredients and directions.",
    "Recipe request\nTitle: {title}\nGive the full recipe.",
    "Task: write a complete recipe.\nDish name: {title}",
]

INGREDIENTS_TO_RECIPE_TEMPLATES = [
    "Got {ingredients} in the fridge — what can I make?",
    "I have {ingredients} on hand. Any recipe ideas?",
    "All I've got is {ingredients}. What can I cook?",
    "Looking at {ingredients} in my kitchen — turn it into a recipe?",
    "Make a recipe using {ingredients}.",
    "Turn these ingredients into a complete recipe: {ingredients}.",
    "Use {ingredients} to write a full recipe.",
    "Build a dish around {ingredients} and give me the recipe.",
    "What can I cook with {ingredients}?",
    "What dish can I make from {ingredients}?",
    "Any idea what to make with {ingredients}?",
    "Suggest a dish I can make with {ingredients}?",
    "Available ingredients:\n{ingredients_block}\nWrite a complete recipe.",
    "Ingredients:\n{ingredients_block}\nSuggest a dish and give the full recipe.",
    "I have these items:\n{ingredients_block}\nGive me a recipe (you may add what's missing).",
]

TITLE_INGREDIENTS_TO_RECIPE_TEMPLATES = [
    "I want to make {title} and I've got {ingredients}. Can you write the recipe?",
    "Making {title} tonight with {ingredients} — how do I do it?",
    "I've got {ingredients} and I'm thinking {title}. Full recipe?",
    "Write a full recipe for {title} using {ingredients}.",
    "Create a recipe for {title} with {ingredients}.",
    "Use {ingredients} to cook {title} — write it out.",
    "Make {title} from {ingredients} and give the complete recipe.",
    "How do I make {title} with {ingredients}?",
    "Can you turn {ingredients} into {title}?",
    "What's the recipe for {title} if I have {ingredients}?",
    "I want to cook {title}. I have {ingredients} — add anything else that's needed and give the recipe.",
    "Help me make {title} with {ingredients}.",
    "Title: {title}\nIngredients:\n{ingredients_block}\nWrite the full recipe.",
    "Dish: {title}\nAvailable:\n{ingredients_block}\nGive ingredients and directions.",
    "Recipe request\nTitle: {title}\nI have:\n{ingredients_block}\nComplete the recipe.",
]

TEMPLATES = {
    "title_to_recipe":             TITLE_TO_RECIPE_TEMPLATES,
    "ingredients_to_recipe":       INGREDIENTS_TO_RECIPE_TEMPLATES,
    "title_ingredients_to_recipe": TITLE_INGREDIENTS_TO_RECIPE_TEMPLATES,
}

for task, tmpls in TEMPLATES.items():
    print(f"{task:30s}: {len(tmpls)} Templates")

title_to_recipe               : 15 Templates
ingredients_to_recipe         : 15 Templates
title_ingredients_to_recipe   : 15 Templates


#### Hilfsfunktionen

- parse_list: die Spalten ingredients, directions, NER liegen als JSON-String-Listen vor.
- format_recipe_output: baut den Output im Pre-Training-Format (corpus.txt-Stil, mehrzeilig).
  Das <|endoftext|>-Stopptoken wird nicht hier gesetzt – das übernimmt im Lab 7 die custom_collate_fn.
- drop_ingredients: zieht p ~ Uniform(0, DROP_P_MAX), lässt jede Kern-Zutat mit Wahrscheinlichkeit p weg
  (mindestens 1 bleibt erhalten) und mischt die verbleibenden, damit das Modell die Reihenfolge
  nicht einfach durchreicht.
- phi3_token_len: Länge des vollständigen Phi-3-Texts (User+Assistant) zum Filtern auf `MAX_TOKENS`.

In [ ]:
def parse_list(cell):
    """JSON-String-Liste -> Liste bereinigter Strings (oder None bei Fehler/leer)."""
    try:
        items = json.loads(cell)
    except (TypeError, ValueError):
        return None
    items = [str(x).strip() for x in items if str(x).strip()]
    return items or None


def format_recipe_output(title, ingredients, directions):
    """Output im Pre-Training-Format (corpus.txt-Stil, mehrzeilig). Ohne <|endoftext|>."""
    ing = "\n".join(ingredients)
    dirs = "\n".join(directions)
    return f"Recipe: {title}\nIngredients:\n{ing}\nDirections:\n{dirs}"


def drop_ingredients(ner, rng):
    """Zufälliger Ingredient-Drop. Gibt (kept_shuffled, p_drop, n_dropped) zurueck."""
    p = rng.uniform(0.0, DROP_P_MAX)
    kept = [x for x in ner if rng.random() > p]
    if not kept:                      # Floor: mindestens eine Zutat bleibt
        kept = [rng.choice(ner)]
    rng.shuffle(kept)                 # Reihenfolge im Prompt aufbrechen
    return kept, round(p, 3), len(ner) - len(kept)


def phi3_token_len(instruction, response):
    """Tokenlaenge des vollstaendigen Phi-3-Texts (User + Assistant)."""
    text = f"<|user|>\n{instruction}\n\n<|assistant|>\n{response}"
    return len(tokenizer.encode(text))


def make_example(row, task, rng):
    """Baut ein Instruction-Beispiel aus einer RecipeNLG-Zeile. None = Zeile ueberspringen."""
    title       = str(row.get("title", "")).strip()
    ingredients = parse_list(row.get("ingredients"))
    directions  = parse_list(row.get("directions"))
    ner         = parse_list(row.get("NER"))

    if not (title and ingredients and directions):
        return None

    response = format_recipe_output(title, ingredients, directions)
    p_drop, n_dropped = 0.0, 0

    if task == "title_to_recipe":
        instruction = rng.choice(TEMPLATES[task]).format(title=title)
    else:
        if not ner:                   # Zutaten-Tasks brauchen NER-Kernzutaten
            return None
        kept, p_drop, n_dropped = drop_ingredients(ner, rng)
        instruction = rng.choice(TEMPLATES[task]).format(
            title=title,
            ingredients=", ".join(kept),
            ingredients_block="\n".join(kept),
        )

    return {
        "conversations": [instruction, response],
        "task": task,
        "p_drop": p_drop,
        "n_dropped": n_dropped,
    }


# Kurzer Funktionscheck an einer Beispielzeile
if df is not None:
    _rng = random.Random(0)
    for _t in TASKS:
        _ex = make_example(df.iloc[0].to_dict(), _t, _rng)
        print(f"=== {_t} (tokens={phi3_token_len(*_ex['conversations'])}) ===")
        print(_ex["conversations"][0])
        print("---")
        print(_ex["conversations"][1][:200], "...\n")

=== title_to_recipe (tokens=191) ===
Recipe request
Title: No-Bake Nut Cookies
Give the full recipe.
---
Recipe: No-Bake Nut Cookies
Ingredients:
1 c. firmly packed brown sugar
1/2 c. evaporated milk
1/2 tsp. vanilla
1/2 c. broken nuts (pecans)
2 Tbsp. butter or margarine
3 1/2 c. bite size shredded rice ...

=== ingredients_to_recipe (tokens=194) ===
What can I cook with vanilla, brown sugar, bite size shredded rice biscuits, nuts, butter?
---
Recipe: No-Bake Nut Cookies
Ingredients:
1 c. firmly packed brown sugar
1/2 c. evaporated milk
1/2 tsp. vanilla
1/2 c. broken nuts (pecans)
2 Tbsp. butter or margarine
3 1/2 c. bite size shredded rice ...

=== title_ingredients_to_recipe (tokens=214) ===
Recipe request
Title: No-Bake Nut Cookies
I have:
bite size shredded rice biscuits
nuts
brown sugar
vanilla
milk
butter
Complete the recipe.
---
Recipe: No-Bake Nut Cookies
Ingredients:
1 c. firmly packed brown sugar
1/2 c. evaporated milk
1/2 tsp. vanilla
1/2 c. broken nuts (pecans)
2 Tbsp. but

#### Datensatz bauen, splitten und als JSONL schreiben

Wir mischen die Zeilen, weisen jeder Zeile reihum einen der drei Tasks zu (→ ~ausgewogen),
bauen das Beispiel, filtern auf MAX_TOKENS und sammeln bis N_TRAIN + N_VAL + N_TEST erreicht ist.
Anschließend wird der Pool gemischt und in Train / Val / Test gesplittet.

Train und Val nutzen das Lab-7-Format `{"conversations": [user, assistant]}` und lassen sich
dort über `format_input_phi3` / `InstructionDataset` einlesen.

Das **Test-Set** enthält nur den User-Prompt (`{"conversations": [user]}`) — beim Inferenz-Test
gibt es keine vorgegebene Referenzantwort; das Modell soll die Antwort selbst generieren.

Zusätzliche Felder (task, p_drop, n_dropped) werden von Lab 7 ignoriert, helfen aber bei der Auswertung.

In [ ]:
def build_dataset(df, n_total, max_tokens, seed=SEED):
    """Erzeugt bis zu n_total laengen-gefilterte, ~ausgewogene Beispiele."""
    rng = random.Random(seed)
    records = df.to_dict("records")
    order = list(range(len(records)))
    rng.shuffle(order)

    pool, n_skipped, n_toolong = [], 0, 0
    for i, ridx in enumerate(order):
        if len(pool) >= n_total:
            break
        task = TASKS[i % len(TASKS)]            # reihum -> ausgewogen
        ex = make_example(records[ridx], task, rng)
        if ex is None:
            n_skipped += 1
            continue
        if phi3_token_len(*ex["conversations"]) > max_tokens:
            n_toolong += 1
            continue
        pool.append(ex)

    rng.shuffle(pool)
    print(f"Gesammelt: {len(pool)} | uebersprungen: {n_skipped} | zu lang (>{max_tokens}): {n_toolong}")
    if len(pool) < n_total:
        print(f"WARNUNG: nur {len(pool)} < {n_total} Beispiele — mehr Zeilen laden (full_dataset.csv).")
    return pool


assert df is not None, "df ist nicht geladen — bitte zuerst die Lade-Zelle ausfuehren."

TOTAL = N_TRAIN + N_VAL + N_TEST
pool  = build_dataset(df, TOTAL, MAX_TOKENS, seed=SEED)

train_data = pool[:N_TRAIN]
val_data   = pool[N_TRAIN:N_TRAIN + N_VAL]
test_data  = pool[N_TRAIN + N_VAL:N_TRAIN + N_VAL + N_TEST]


def write_jsonl(path, data):
    with open(path, "w", encoding="utf-8") as f:
        for ex in data:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")


def to_test_entry(ex):
    """Test-Set: nur User-Prompt, keine Referenz-Antwort (Inference-Setting)."""
    return {**ex, "conversations": [ex["conversations"][0]]}


os.makedirs(OUT_DIR, exist_ok=True)
suffix = f"ctx{MAX_TOKENS}_seed{SEED}"
paths = {
    "train": f"{OUT_DIR}/recipe_instruct_train_{suffix}.jsonl",
    "val":   f"{OUT_DIR}/recipe_instruct_val_{suffix}.jsonl",
    "test":  f"{OUT_DIR}/recipe_instruct_test_{suffix}.jsonl",
}
write_jsonl(paths["train"], train_data)
write_jsonl(paths["val"],   val_data)
write_jsonl(paths["test"],  [to_test_entry(ex) for ex in test_data])

print(f"\nTrain: {len(train_data)} -> {paths['train']}")
print(f"Val:   {len(val_data)} -> {paths['val']}")
print(f"Test:  {len(test_data)} -> {paths['test']}")

Gesammelt: 17000 | uebersprungen: 1 | zu lang (>512): 2

Train: 15000 -> datasets/instruction/recipe_instruct_train_ctx512_seed123.jsonl
Val:   1000 -> datasets/instruction/recipe_instruct_val_ctx512_seed123.jsonl
Test:  1000 -> datasets/instruction/recipe_instruct_test_ctx512_seed123.jsonl


#### Kontrolle: Verteilung & Beispiele

Task-Balance, Tokenlängen-Statistik und je ein vollständiges Beispiel pro Task. Beim
`ingredients_to_recipe`-Beispiel sieht man den Drop: Im Prompt fehlen Kern-Zutaten, die im
Output trotzdem (mengenbehaftet) auftauchen.

In [ ]:
from collections import Counter

# Task-Balance (Train)
counts = Counter(ex["task"] for ex in train_data)
print("Task-Verteilung (Train):")
for t in TASKS:
    print(f"  {t:30s}: {counts[t]:>6d}  ({counts[t]/len(train_data):.1%})")

# Tokenlaengen-Statistik
lengths = sorted(phi3_token_len(*ex["conversations"]) for ex in train_data)
n = len(lengths)
print(f"\nTokenlaengen (Train, MAX_TOKENS={MAX_TOKENS}):")
print(f"  min={lengths[0]}  median={lengths[n//2]}  p90={lengths[int(n*0.9)]}  "
      f"p99={lengths[int(n*0.99)]}  max={lengths[-1]}")

# Drop-Statistik (nur Zutaten-Tasks)
drops = [ex["n_dropped"] for ex in train_data if ex["task"] != "title_to_recipe"]
if drops:
    print(f"\nIngredient-Drop (Zutaten-Tasks): mittl. weggelassen={sum(drops)/len(drops):.2f}, "
          f"max={max(drops)}, Anteil mit >=1 Drop={sum(d>0 for d in drops)/len(drops):.1%}")

# Je ein vollstaendiges Beispiel pro Task
print("\n" + "=" * 70)
seen = set()
for ex in train_data:
    t = ex["task"]
    if t in seen:
        continue
    seen.add(t)
    instr, resp = ex["conversations"]
    print(f"\n##### {t}  (p_drop={ex['p_drop']}, n_dropped={ex['n_dropped']})")
    print("----- USER -----")
    print(instr)
    print("----- ASSISTANT -----")
    print(resp)
    if len(seen) == len(TASKS):
        break

Task-Verteilung (Train):
  title_to_recipe               :   5052  (33.7%)
  ingredients_to_recipe         :   4944  (33.0%)
  title_ingredients_to_recipe   :   5004  (33.4%)

Tokenlaengen (Train, MAX_TOKENS=512):
  min=61  median=164  p90=237  p99=318  max=489

Ingredient-Drop (Zutaten-Tasks): mittl. weggelassen=1.80, max=12, Anteil mit >=1 Drop=72.9%


##### ingredients_to_recipe  (p_drop=0.017, n_dropped=0)
----- USER -----
All I've got is shrimp, mayo, onion, cream cheese, lemon juice, celery, salt. What can I cook?
----- ASSISTANT -----
Recipe: Country Cafe'S Shrimp Spread
Ingredients:
8 oz. cream cheese, softened
1/2 c. mayo
1/4 c. chopped onion
1 rib chopped celery
1 tsp. lemon juice
1 small can shrimp, drained
salt and pepper to taste
Directions:
Mix all of the ingredients with fork.
Chill.
Serve with crackers.

##### title_ingredients_to_recipe  (p_drop=0.112, n_dropped=0)
----- USER -----
Title: Vanilla Ice Cream
Ingredients:
milk
milk
condensed milk
sugar
salt
vanilla
eggs
W